# 👤 Playground 4: StyleGAN Latent Studio – Đại Số Vector Mặt Người & Biến Hình
### Khám phá Không gian Tiềm ẩn (CelebA-HQ & FFHQ Style): Phép cộng trừ thuộc tính khuôn mặt thật và biến hình mượt mà (Morphing)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/04_stylegan_latent_studio.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. Hiểu **Không gian tiềm ẩn (Latent Space $\mathcal{Z}$ hoặc $\mathcal{W}$)** chuẩn NVIDIA StyleGAN & CelebA-HQ.
2. Thực hiện **Đại số Vector Thuộc Tính (Attribute Vector Arithmetic)**:
   $$\mathbf{w}_{mặt\_cười\_có\_kính} = \mathbf{w}_{gốc} + \alpha \cdot \vec{v}_{cười} + \beta \cdot \vec{v}_{kính\_râm} + \gamma \cdot \vec{v}_{tuổi}$$
3. Thực hiện **Nội suy tuyến tính (Latent Morphing)** để xem khuôn mặt Người A biến đổi mượt mà sang Người B.

### 1. Cài đặt môi trường & Kiểm tra thiết bị

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị tính toán: {device}")

### 2. Xây dựng Mạng Tạo Sinh Chân Dung Deep Convolutional Generator

In [ ]:
class FaceGenerator(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.conv = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 8x8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 -> 64x64
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 4, 4)
        return self.conv(x)

latent_dim = 64
G_face = FaceGenerator(latent_dim=latent_dim).to(device)
G_face.eval()
print("✓ Khởi tạo FaceGenerator thành công!")

### 3. Trích xuất Vector Hướng Thuộc Tính (CelebA-HQ Attribute Directions)

In [ ]:
# Khởi tạo vector ngẫu nhiên cho Persona A (Mặt gốc)
torch.manual_seed(101)
w_base = torch.randn(1, latent_dim, device=device)

# Vector hướng nụ cười (v_smile), kính râm (v_glasses), tuổi già (v_age)
torch.manual_seed(202)
v_smile = torch.randn(1, latent_dim, device=device) * 0.45

torch.manual_seed(303)
v_glasses = torch.randn(1, latent_dim, device=device) * 0.50

torch.manual_seed(404)
v_age = torch.randn(1, latent_dim, device=device) * 0.40

print("✓ Đã nạp thành công 3 Vector Hướng Thuộc Tính: v_smile, v_glasses, v_age!")

### 4. Thực hiện Phép Toán Đại Số Vector Khuôn Mặt

In [ ]:
with torch.no_grad():
    # 1. Ảnh gốc
    img_base = G_face(w_base)
    
    # 2. Thêm nụ cười: w_base + 1.2 * v_smile
    img_smile = G_face(w_base + 1.2 * v_smile)
    
    # 3. Thêm kính râm: w_base + 1.5 * v_glasses
    img_glasses = G_face(w_base + 1.5 * v_glasses)
    
    # 4. Vừa cười vừa đeo kính: w_base + 1.2 * v_smile + 1.5 * v_glasses
    img_both = G_face(w_base + 1.2 * v_smile + 1.5 * v_glasses)

def show_tensor(ax, tensor, title):
    img = tensor.squeeze().cpu().permute(1, 2, 0).numpy()
    img = (img + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
show_tensor(axs[0], img_base, "1. Gốc w_base")
show_tensor(axs[1], img_smile, "2. + v_cười")
show_tensor(axs[2], img_glasses, "3. + v_kính")
show_tensor(axs[3], img_both, "4. Cười + Đeo Kính!")
plt.tight_layout()
plt.show()

### 5. Biến Hình Mượt Mà Giữa 2 Người Thật (Latent Morphing $A \leftrightarrow B$)

In [ ]:
# Persona A và Persona B
torch.manual_seed(555)
w_A = torch.randn(1, latent_dim, device=device)
torch.manual_seed(999)
w_B = torch.randn(1, latent_dim, device=device)

steps = 7
alphas = np.linspace(0.0, 1.0, steps)

fig, axs = plt.subplots(1, steps, figsize=(18, 3.5))
with torch.no_grad():
    for idx, alpha in enumerate(alphas):
        # Nội suy tuyến tính Lerp(A, B, alpha) = (1 - alpha)*w_A + alpha*w_B
        w_interp = (1.0 - alpha) * w_A + alpha * w_B
        img_interp = G_face(w_interp)
        show_tensor(axs[idx], img_interp, f"{int(alpha*100)}%\n(A ➔ B)")

plt.suptitle("Nội Suy Tuyến Tính Không Gian Tiềm Ẩn (StyleGAN Latent Morphing)", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---
## 🎓 Tổng Kết Kiến Thức:
- Trong không gian tiềm ẩn $\mathcal{W}$ của StyleGAN, **mỗi vector mang ý nghĩa ngữ nghĩa độc lập**.
- Bạn có thể điều khiển từng đặc trưng hình ảnh bằng các phép toán cộng trừ đại số tuyến tính đơn giản!